In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.bronze")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType
from delta.tables import DeltaTable

In [ ]:
df = (
    spark.table("stocks.stage.fx_exchange_rates_raw")
    .withColumn("date",         F.to_date("date"))
    .withColumn("rate",         F.col("rate").cast(DoubleType()))
    .withColumn("ingested_at",  F.current_timestamp())
)

In [ ]:
tbl = "stocks.bronze.fx_exchange_rates"
if not spark.catalog.tableExists(tbl):
    df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    df.alias("s"),
    "s.date = t.date AND s.currency = t.currency"
).whenNotMatchedInsertAll().execute()

print(f"stocks.bronze.fx_exchange_rates: {spark.table(tbl).count()} rows")
spark.table(tbl).groupBy("currency").count().orderBy("currency").show()